# 01 — Main Experiment
**Paper:** *Cross-Dataset Pneumonia Detection Failure as an Ontological Problem*

## Pipeline
1. Load config & data
2. Train CheXpert model
3. Compute semantic distance matrix (BioMedBERT)
4. Zero-shot transfer baseline
5. Label confusion / failure taxonomy
6. Semantic distance ↔ performance correlation
7. Ontology-guided fine-tuning vs standard
8. Recovery rate + alpha ablation

> Run `00_setup.ipynb` first.

In [ ]:
# IMPORTS & CONFIG
import sys, os, json, copy, random
import torch, numpy as np
import matplotlib.pyplot as plt

BASE_DIR = '/content/drive/MyDrive/pneumonia_research'
with open(f'{BASE_DIR}/config.json') as f:
    CONFIG = json.load(f)

REPO_DIR = '/content/pneumonia_research'
sys.path.insert(0, REPO_DIR)

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

torch.manual_seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
random.seed(CONFIG['seed'])
torch.backends.cudnn.deterministic = True
print('Config loaded.')

## Phase 1 — Train CheXpert Model
ResNet50 multi-label classifier on 6 CheXpert labels. Saved to Drive so it only trains once.

In [ ]:
from src.data.datasets import get_chexpert_loaders
from src.models.classifier import CheXpertClassifier
import torch.nn as nn, torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

chexpert_loaders = get_chexpert_loaders(
    chexpert_root=CONFIG['chexpert_dir'],
    image_size=CONFIG['image_size'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
    uncertain_strategy='smooth'
)

In [ ]:
CHEXPERT_CKPT = f'{BASE_DIR}/results/chexpert_model.pt'
os.makedirs(f'{BASE_DIR}/results', exist_ok=True)

if os.path.exists(CHEXPERT_CKPT):
    print('Loading saved CheXpert model...')
    chexpert_model = CheXpertClassifier.load(CHEXPERT_CKPT).to(DEVICE)
else:
    print('Training CheXpert model...')
    chexpert_model = CheXpertClassifier(num_labels=6, pretrained=True).to(DEVICE)
    optimizer  = optim.AdamW(chexpert_model.parameters(), lr=CONFIG['lr'], weight_decay=1e-4)
    scheduler  = CosineAnnealingLR(optimizer, T_max=CONFIG['epochs'])
    criterion  = nn.BCEWithLogitsLoss()

    for epoch in range(CONFIG['epochs']):
        chexpert_model.train()
        losses = []
        for batch in tqdm(chexpert_loaders['train'], desc=f'Epoch {epoch+1}/{CONFIG["epochs"]}'):
            if batch is None: continue
            imgs, labels, _ = batch
            imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(chexpert_model(imgs), labels)
            loss.backward()
            optimizer.step()
            losses.append(loss.item())
        scheduler.step()
        print(f'  Epoch {epoch+1}: loss={np.mean(losses):.4f}')

    chexpert_model.save(CHEXPERT_CKPT)
    print('CheXpert model saved.')

## Phase 2 — Semantic Distance Matrix
BioMedBERT embeddings for all labels. **Figure 1 & 2** of the paper.

In [ ]:
from src.analysis.semantic_alignment import (
    BioMedBERTEncoder, SemanticDistanceMatrix
)

SEMANTIC_DIR = f'{BASE_DIR}/results/semantic'
os.makedirs(SEMANTIC_DIR, exist_ok=True)

if os.path.exists(f'{SEMANTIC_DIR}/similarity_matrix.npy'):
    print('Loading precomputed semantic matrices...')
    distance_matrix = SemanticDistanceMatrix().load(SEMANTIC_DIR)
else:
    print('Computing BioMedBERT embeddings (~2 mins)...')
    encoder = BioMedBERTEncoder(device=DEVICE)
    distance_matrix = SemanticDistanceMatrix(encoder=encoder)
    distance_matrix.compute(save_path=SEMANTIC_DIR)

print('\nDistances: CheXpert labels -> Kaggle/Pneumonia')
for label, dist in distance_matrix.get_chexpert_to_kaggle_distances().items():
    print(f'  {label:<35} {dist:.4f}')

In [ ]:
# Figure 1 — Heatmap
fig1 = distance_matrix.plot_heatmap(
    save_path=f'{SEMANTIC_DIR}/fig1_similarity_heatmap.png')
plt.show()

# Figure 2 — Distance bar chart
fig2 = distance_matrix.plot_distance_bar(
    save_path=f'{SEMANTIC_DIR}/fig2_distance_bar.png')
plt.show()

## Phase 3 — Zero-Shot Transfer Baseline
CheXpert model on Kaggle test set with no adaptation.

In [ ]:
from src.data.datasets import get_kaggle_loaders
from src.models.classifier import MultiOutputWrapper
from sklearn.metrics import roc_auc_score, f1_score, classification_report

kaggle_loaders = get_kaggle_loaders(
    kaggle_root=CONFIG['kaggle_dir'],
    image_size=CONFIG['image_size'],
    batch_size=CONFIG['batch_size'],
    num_workers=CONFIG['num_workers'],
)

wrapped = MultiOutputWrapper(chexpert_model).to(DEVICE)
wrapped.eval()

all_probs, all_labels = [], []
with torch.no_grad():
    for imgs, labels, _ in tqdm(kaggle_loaders['test'], desc='Zero-shot eval'):
        probs = wrapped.predict_pneumonia_prob(imgs.to(DEVICE)).cpu().numpy()
        all_probs.extend(probs)
        all_labels.extend(labels.numpy())

all_probs  = np.array(all_probs)
all_labels = np.array(all_labels)
preds      = (all_probs >= 0.5).astype(int)

baseline_metrics = {
    'auc': roc_auc_score(all_labels, all_probs),
    'f1':  f1_score(all_labels, preds, zero_division=0),
}
print(f'ZERO-SHOT  AUC={baseline_metrics["auc"]:.4f}  F1={baseline_metrics["f1"]:.4f}')
print(classification_report(all_labels, preds, target_names=['Normal','Pneumonia']))

## Phase 4 — Label Confusion Analysis
The clinical dialect finding. **Figure 3 & Table 2**.

In [ ]:
from src.analysis.label_confusion import (
    PredictionCollector, FailureTaxonomy, compute_label_confusion_matrix
)

CONFUSION_DIR = f'{BASE_DIR}/results/label_confusion'
os.makedirs(CONFUSION_DIR, exist_ok=True)

collector = PredictionCollector(chexpert_model, device=DEVICE)
predictions_df = collector.collect(kaggle_loaders['test'])
predictions_df.to_csv(f'{CONFUSION_DIR}/predictions.csv', index=False)
print(f'Collected {len(predictions_df)} predictions')
predictions_df.head()

In [ ]:
taxonomy = FailureTaxonomy(predictions_df)
taxonomy.analyze().report()

fig3a = taxonomy.plot_taxonomy_pie(save_path=f'{CONFUSION_DIR}/fig3a_failure_taxonomy.png')
plt.show()

fig3b = taxonomy.plot_label_activation_heatmap(save_path=f'{CONFUSION_DIR}/fig3b_activation_heatmap.png')
plt.show()

confusion_df, fig_t2 = compute_label_confusion_matrix(
    predictions_df, save_path=f'{CONFUSION_DIR}/table2_label_confusion.png')
plt.show()

## Phase 5 — Semantic Distance ↔ Performance Correlation
**Figure 4 & Table 3** — the paper's core empirical claim.

In [ ]:
from src.analysis.semantic_alignment import correlate_distance_with_performance

CORR_DIR = f'{BASE_DIR}/results/correlation'
os.makedirs(CORR_DIR, exist_ok=True)

label_performance = {}
chexpert_labels = ['Pneumonia','Lung Opacity','Consolidation',
                   'Edema','Pleural Effusion','Atelectasis']

for label in chexpert_labels:
    col = f'prob_{label}'
    if col in predictions_df.columns:
        probs = predictions_df[col].values
        true  = predictions_df['true_label'].values
        try:
            auc = roc_auc_score(true, probs)
        except:
            auc = 0.5
        f1 = f1_score(true, (probs >= 0.5).astype(int), zero_division=0)
        label_performance[f'CheXpert/{label}'] = {'auc': auc, 'f1': f1}
        print(f'  CheXpert/{label:<22} AUC={auc:.4f}  F1={f1:.4f}')

correlation_results, fig4 = correlate_distance_with_performance(
    distance_matrix, label_performance,
    save_path=f'{CORR_DIR}/fig4_distance_vs_performance.png')
plt.show()

with open(f'{CORR_DIR}/correlation_results.json', 'w') as f:
    json.dump(correlation_results, f, indent=2)

## Phase 6 — Ontology-Guided Fine-Tuning
**Figure 5 & Table 4** — proposed fix vs standard baseline.

In [ ]:
from src.models.classifier import KaggleAdaptedClassifier
from src.training.ontology_finetuning import (
    OntologyGuidedLoss, OntologyFineTuner,
    compute_ontology_weights, compute_recovery_rate
)

FINETUNE_DIR = f'{BASE_DIR}/results/finetuning'
os.makedirs(FINETUNE_DIR, exist_ok=True)

ALPHA = 1.0
ontology_weights = compute_ontology_weights(
    distance_matrix, chexpert_labels,
    target_label='Kaggle/Pneumonia', alpha=ALPHA)

In [ ]:
# Standard fine-tuning
print('--- Standard BCE Fine-Tuning ---')
model_standard = KaggleAdaptedClassifier(
    chexpert_model=copy.deepcopy(chexpert_model)).to(DEVICE)

tuner_std = OntologyFineTuner(model_standard, device=DEVICE)
opt_std   = torch.optim.AdamW(model_standard.parameters(), lr=1e-4)
bce_loss  = nn.BCEWithLogitsLoss()

history_std = tuner_std.finetune(
    kaggle_loaders['train'], kaggle_loaders['val'],
    loss_fn=lambda logits, targets, paths: bce_loss(logits, targets.float()),
    optimizer=opt_std, epochs=10, label='standard')

torch.save(model_standard.state_dict(), f'{FINETUNE_DIR}/model_standard.pt')

In [ ]:
# Ontology-guided fine-tuning
print('--- Ontology-Guided Fine-Tuning ---')
model_ogft = KaggleAdaptedClassifier(
    chexpert_model=copy.deepcopy(chexpert_model)).to(DEVICE)

ogft_loss  = OntologyGuidedLoss(
    label_weights={'Pneumonia': ontology_weights['Pneumonia']},
    alpha=ALPHA).to(DEVICE)
opt_ogft   = torch.optim.AdamW(model_ogft.parameters(), lr=1e-4)
tuner_ogft = OntologyFineTuner(model_ogft, device=DEVICE)

history_ogft = tuner_ogft.finetune(
    kaggle_loaders['train'], kaggle_loaders['val'],
    loss_fn=lambda logits, targets, paths: ogft_loss(
        logits.unsqueeze(-1), targets.float().unsqueeze(-1)),
    optimizer=opt_ogft, epochs=10, label='ontology')

torch.save(model_ogft.state_dict(), f'{FINETUNE_DIR}/model_ogft.pt')

## Phase 7 — Final Results & Recovery Rate

In [ ]:
evaluator = OntologyFineTuner(model_standard, device=DEVICE)
std_metrics  = evaluator.evaluate(kaggle_loaders['test'])
evaluator.model = model_ogft
ogft_metrics = evaluator.evaluate(kaggle_loaders['test'])

print(f'\n{"Method":<28} {"AUC":>8} {"F1":>8} {"Acc":>8} {"ECE":>8}')
print('-' * 58)
print(f'{"Zero-shot baseline":<28} {baseline_metrics["auc"]:>8.4f} {baseline_metrics["f1"]:>8.4f}')
print(f'{"Standard fine-tuning":<28} {std_metrics["auc"]:>8.4f} {std_metrics["f1"]:>8.4f} {std_metrics["accuracy"]:>8.4f} {std_metrics["ece"]:>8.4f}')
print(f'{"OGFT (ours)":<28} {ogft_metrics["auc"]:>8.4f} {ogft_metrics["f1"]:>8.4f} {ogft_metrics["accuracy"]:>8.4f} {ogft_metrics["ece"]:>8.4f}')

recovery = compute_recovery_rate(baseline_metrics, std_metrics, ogft_metrics, metric='auc')

tuner_std.history = {'standard': history_std, 'ontology': history_ogft}
fig5 = tuner_std.plot_comparison(save_path=f'{BASE_DIR}/results/fig5_finetuning_curves.png')
plt.show()

final = {'baseline': baseline_metrics, 'standard': std_metrics,
         'ogft': ogft_metrics, 'recovery': recovery, 'correlation': correlation_results}
with open(f'{BASE_DIR}/results/final_results.json', 'w') as f:
    json.dump(final, f, indent=2)
print('All results saved.')

In [ ]:
# Alpha ablation study
print('Running alpha ablation...')
alphas = [0.0, 0.25, 0.5, 0.75, 1.0, 1.5, 2.0]
ablation = []

for alpha in alphas:
    m = KaggleAdaptedClassifier(chexpert_model=copy.deepcopy(chexpert_model)).to(DEVICE)
    w = compute_ontology_weights(distance_matrix, chexpert_labels,
                                  target_label='Kaggle/Pneumonia', alpha=alpha)
    loss_a = OntologyGuidedLoss(label_weights={'Pneumonia': w['Pneumonia']}, alpha=alpha).to(DEVICE)
    t = OntologyFineTuner(m, device=DEVICE)
    o = torch.optim.AdamW(m.parameters(), lr=1e-4)
    t.finetune(kaggle_loaders['train'], kaggle_loaders['val'],
               loss_fn=lambda logits, targets, paths: loss_a(
                   logits.unsqueeze(-1), targets.float().unsqueeze(-1)),
               optimizer=o, epochs=5, label='ontology')
    met = t.evaluate(kaggle_loaders['test'])
    ablation.append({'alpha': alpha, **met})
    print(f'  alpha={alpha}: AUC={met["auc"]:.4f}')

fig_abl, ax = plt.subplots(figsize=(8, 5))
ax.plot([r['alpha'] for r in ablation], [r['auc'] for r in ablation],
        'o-', lw=2.5, ms=8, color='#FF5722', label='OGFT')
ax.axhline(baseline_metrics['auc'], color='gray',  ls='--', label='Zero-shot')
ax.axhline(std_metrics['auc'],      color='blue',  ls='--', label='Standard FT')
ax.set_xlabel('Alpha', fontsize=12)
ax.set_ylabel('AUC-ROC', fontsize=12)
ax.set_title('Alpha Ablation Study', fontsize=13, fontweight='bold')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{BASE_DIR}/results/fig_ablation_alpha.png', dpi=300, bbox_inches='tight')
plt.show()
print('Ablation complete.')